In [49]:
import sys
from pathlib import Path

repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root))


# Study Question


This notebook compares a small set of candidate tendon routing geometries for a simplified index-finger model. The goal is to decide which routing family should be carried forward to the first prototype concept, using tendon excursion and torque per unit tendon tension as the main comparison outputs.


In [50]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go


from simulation_modeling import v0_model

In [51]:
# CONSTANTS

geom = v0_model.FingerGeometry(50, 30, 22)  # mm
neutral_q = [0.0, 0.0, 0.0]
pip_scale = 25.0 / 40.0
dip_scale = 10.0 / 25.0
q_sweep_rad = v0_model.coordinated_flexion_sweep(
    n=41,
    mcp_range_deg=(0.0, -40.0),
    pip_scale=pip_scale,
    dip_scale=dip_scale,
)
q_sweep_deg = np.rad2deg(q_sweep_rad)

length_scale_m = 1e-3  # convert mm-based model outputs when combining with N m torques
reference_tension = 1.0  # N, used for torque-per-unit-tension plots
torque_target_range_nm = (0.05, 0.45)
gradient_floor = 1e-6

print(q_sweep_deg[0])
print(q_sweep_deg[-1])
print(q_sweep_deg.shape)


[0. 0. 0.]
[-40. -25. -10.]
(41, 3)


In [52]:
fk_neutral = v0_model.forward_kinematics(geom, q_sweep_rad[0])
fk_flexed = v0_model.forward_kinematics(geom, q_sweep_rad[-1])

print(fk_neutral.Fingertip)
print(fk_flexed.Fingertip)



[87.  0.]
[ 47.57227788 -69.23681099]


In [53]:
# Candidate routings

x_entry = -20.0

def make_candidate_path(name, h, prox_u=None, mid_u=None, dist_u=5.0):
    elements = [
        v0_model.RoutingElement("entry", "world", "entry", np.array([x_entry, -h], dtype=float)),
    ]
    if prox_u is not None:
        elements.append(
            v0_model.RoutingElement("prox_guide", "proximal", "guide", np.array([prox_u, -h], dtype=float))
        )
    if mid_u is not None:
        elements.append(
            v0_model.RoutingElement("mid_guide", "middle", "guide", np.array([mid_u, -h], dtype=float))
        )
    elements.append(
        v0_model.RoutingElement("dist_anchor", "distal", "anchor", np.array([dist_u, -h], dtype=float))
    )
    return v0_model.RoutingPath(name, tuple(elements))

candidate_paths = {
    "V0_thimble_only": make_candidate_path("V0_thimble_only", h=5.0, prox_u=None, mid_u=None, dist_u=5.0),
    "V1_baseline_two_guides": make_candidate_path("V1_baseline_two_guides", h=5.0, prox_u=5.0, mid_u=5.0, dist_u=5.0),
    "V2_high_offset_two_guides": make_candidate_path("V2_high_offset_two_guides", h=9.0, prox_u=5.0, mid_u=5.0, dist_u=5.0),
    "V3_proximal_bias": make_candidate_path("V3_proximal_bias", h=5.0, prox_u=12.0, mid_u=4.0, dist_u=4.0),
    "V4_low_offset_two_guides": make_candidate_path("V4_low_offset_two_guides", h=2.5, prox_u=5.0, mid_u=5.0, dist_u=5.0),
}

list(candidate_paths)


['V0_thimble_only',
 'V1_baseline_two_guides',
 'V2_high_offset_two_guides',
 'V3_proximal_bias',
 'V4_low_offset_two_guides']

In [54]:
# Check one candidate at two postures

baseline_path = candidate_paths["V1_baseline_two_guides"]

L_neutral, seg_neutral, pts_neutral = v0_model.tendon_path_length(fk_neutral, baseline_path)
L_flexed, seg_flexed, pts_flexed = v0_model.tendon_path_length(fk_flexed, baseline_path)

print("neutral length:", L_neutral)
print("flexed length:", L_flexed)
print("excursion:", L_flexed - L_neutral)
print("neutral points:", pts_neutral)
print("flexed points:", pts_flexed)

neutral length: 92.0
flexed length: 84.23933270602994
excursion: -7.760667293970059
neutral points: [[-20.  -5.]
 [  5.  -5.]
 [ 46.  -5.]
 [ 72.  -5.]]
flexed points: [[-20.          -5.        ]
 [  0.61628417  -7.04416026]
 [ 28.98937454 -32.99892224]
 [ 38.86036307 -56.04201882]]


In [55]:
# Evaluate all candidates over the shared posture sweep

candidate_results = {
    name: v0_model.evaluate_path_over_sweep(geom, path, q_sweep_rad)
    for name, path in candidate_paths.items()
}

candidate_results["V1_baseline_two_guides"].keys()


dict_keys(['q', 'L', 'dL', 'grad', 'tau_T1'])

In [56]:
# Summary metrics per candidate

def count_sign_flips(values, tol=1e-9):
    filtered = []
    for value in values:
        if np.abs(value) <= tol:
            continue
        filtered.append(np.sign(value))
    if len(filtered) < 2:
        return 0
    return int(np.sum(np.diff(filtered) != 0))

summary_rows = []

for name, result in candidate_results.items():
    path = candidate_paths[name]
    grad_abs_mm = np.abs(result["grad"])
    grad_abs_m = grad_abs_mm * length_scale_m
    mcp_abs_grad_m = grad_abs_m[:, 0]
    safe_mcp_grad_m = np.where(mcp_abs_grad_m > gradient_floor, mcp_abs_grad_m, np.nan)
    tension_low_n = torque_target_range_nm[0] / safe_mcp_grad_m
    tension_high_n = torque_target_range_nm[1] / safe_mcp_grad_m
    joint_share = grad_abs_mm.mean(axis=0)
    joint_share = joint_share / joint_share.sum()
    excursion_diff = np.diff(result["dL"])
    complexity_score = sum(element.kind == "guide" for element in path.elements) + 0.5 * max(len(path.elements) - 2, 0)
    summary_rows.append(
        {
            "candidate": name,
            "guide_count": sum(element.kind == "guide" for element in path.elements),
            "routing_point_count": len(path.elements),
            "complexity_score": complexity_score,
            "neutral_length_mm": result["L"][0],
            "final_length_mm": result["L"][-1],
            "total_excursion_mm": result["dL"][-1],
            "excursion_monotonic_nonincreasing": bool(np.all(excursion_diff <= 1e-9)),
            "excursion_sign_flips": count_sign_flips(np.diff(result["dL"])),
            "mcp_torque_sign_flips": count_sign_flips(result["tau_T1"][:, 0]),
            "peak_abs_dL_dq_m_per_rad": grad_abs_m.max(),
            "min_abs_dL_dq_mcp_m_per_rad": np.nanmin(safe_mcp_grad_m),
            "mean_abs_dL_dq_mcp_m_per_rad": grad_abs_m[:, 0].mean(),
            "mean_abs_dL_dq_pip_m_per_rad": grad_abs_m[:, 1].mean(),
            "mean_abs_dL_dq_dip_m_per_rad": grad_abs_m[:, 2].mean(),
            "mcp_share": joint_share[0],
            "pip_share": joint_share[1],
            "dip_share": joint_share[2],
            "joint_balance_std": joint_share.std(),
            "peak_tension_low_nm_target": np.nanmax(tension_low_n),
            "peak_tension_high_nm_target": np.nanmax(tension_high_n),
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values("candidate").reset_index(drop=True)
summary_df

,candidate,guide_count,routing_point_count,complexity_score,neutral_length_mm,final_length_mm,total_excursion_mm,excursion_monotonic_nonincreasing,excursion_sign_flips,mcp_torque_sign_flips,...,min_abs_dL_dq_mcp_m_per_rad,mean_abs_dL_dq_mcp_m_per_rad,mean_abs_dL_dq_pip_m_per_rad,mean_abs_dL_dq_dip_m_per_rad,mcp_share,pip_share,dip_share,joint_balance_std,peak_tension_low_nm_target,peak_tension_high_nm_target
0,V0_thimble_only,0,2,0.0,92.0,77.909114,-14.090886,True,0,0,...,0.0050,0.011375,0.011602,0.006157,0.390438,0.398221,0.211342,0.086320,10.000000,90.0
1,V1_baseline_two_guides,2,4,3.0,92.0,84.239333,-7.760667,True,0,0,...,0.0050,0.006130,0.005834,0.005338,0.354308,0.337185,0.308507,0.018895,10.000000,90.0
2,V2_high_offset_two_guides,2,4,3.0,92.0,79.201245,-12.798755,True,0,0,...,0.0090,0.009906,0.009737,0.009326,0.341935,0.336128,0.321937,0.008400,5.555556,50.0
3,V3_proximal_bias,2,4,3.0,91.0,82.405278,-8.594722,True,0,0,...,0.0050,0.007460,0.005644,0.005279,0.405821,0.307020,0.287159,0.051894,10.000000,90.0
4,V4_low_offset_two_guides,2,4,3.0,92.0,87.401341,-4.598659,True,0,0,...,0.0025,0.003753,0.003392,0.002845,0.375699,0.339512,0.284789,0.037370,20.000000,180.0


In [57]:
# Simple screening view

screen_df = summary_df.copy()
screen_df["abs_excursion_mm"] = screen_df["total_excursion_mm"].abs()
screen_df["screen_rank"] = (
    screen_df["peak_tension_high_nm_target"].rank(method="dense")
    + screen_df["complexity_score"].rank(method="dense")
    + screen_df["joint_balance_std"].rank(method="dense")
    + screen_df["abs_excursion_mm"].rank(method="dense")
)

screen_columns = [
    "candidate",
    "guide_count",
    "complexity_score",
    "abs_excursion_mm",
    "peak_tension_high_nm_target",
    "joint_balance_std",
    "excursion_monotonic_nonincreasing",
    "mcp_torque_sign_flips",
    "screen_rank",
]

screen_df[screen_columns].sort_values(["screen_rank", "candidate"]).reset_index(drop=True)


,candidate,guide_count,complexity_score,abs_excursion_mm,peak_tension_high_nm_target,joint_balance_std,excursion_monotonic_nonincreasing,mcp_torque_sign_flips,screen_rank
0,V2_high_offset_two_guides,2,3.0,12.798755,50.0,0.008400,True,0,8.0
1,V1_baseline_two_guides,2,3.0,7.760667,90.0,0.018895,True,0,9.0
2,V4_low_offset_two_guides,2,3.0,4.598659,180.0,0.037370,True,0,10.0
3,V3_proximal_bias,2,3.0,8.594722,90.0,0.051894,True,0,12.0
4,V0_thimble_only,0,0.0,14.090886,90.0,0.086320,True,0,13.0


In [58]:
# Comparison plots

mcp_deg = np.rad2deg(q_sweep_rad[:, 0])

fig_excursion = go.Figure()
for name, result in candidate_results.items():
    fig_excursion.add_trace(
        go.Scatter(x=mcp_deg, y=result["dL"], mode="lines+markers", name=name)
    )

fig_excursion.update_layout(
    title="Tendon Excursion vs MCP Angle",
    xaxis_title="MCP angle (deg)",
    yaxis_title="Excursion relative to neutral (mm)",
    template="plotly_white",
)

fig_excursion.show()

fig_mcp = go.Figure()
for name, result in candidate_results.items():
    tau_mcp_nm_per_n = result["tau_T1"][:, 0] * length_scale_m
    fig_mcp.add_trace(
        go.Scatter(x=mcp_deg, y=tau_mcp_nm_per_n, mode="lines+markers", name=name)
    )

fig_mcp.update_layout(
    title="MCP Torque per 1 N Tension vs MCP Angle",
    xaxis_title="MCP angle (deg)",
    yaxis_title="Torque per 1 N tension (N m)",
    template="plotly_white",
)

fig_mcp.show()

fig_tension = go.Figure()
for name, result in candidate_results.items():
    mcp_abs_grad_m = np.abs(result["grad"][:, 0]) * length_scale_m
    safe_mcp_grad_m = np.where(mcp_abs_grad_m > gradient_floor, mcp_abs_grad_m, np.nan)
    required_tension_high = torque_target_range_nm[1] / safe_mcp_grad_m
    fig_tension.add_trace(
        go.Scatter(x=mcp_deg, y=required_tension_high, mode="lines+markers", name=name)
    )

fig_tension.update_layout(
    title="Required Tendon Tension for 0.45 Nm MCP Torque",
    xaxis_title="MCP angle (deg)",
    yaxis_title="Required tendon tension (N)",
    template="plotly_white",
)

fig_tension.show()

fig_joint_share = go.Figure()
fig_joint_share.add_trace(go.Bar(x=summary_df["candidate"], y=summary_df["mcp_share"], name="MCP share"))
fig_joint_share.add_trace(go.Bar(x=summary_df["candidate"], y=summary_df["pip_share"], name="PIP share"))
fig_joint_share.add_trace(go.Bar(x=summary_df["candidate"], y=summary_df["dip_share"], name="DIP share"))
fig_joint_share.update_layout(
    title="Mean Joint Leverage Share by Candidate",
    xaxis_title="Candidate",
    yaxis_title="Fraction of mean absolute gradient",
    barmode="group",
    template="plotly_white",
)

fig_joint_share.show()
